# Data Cleaning, Step by Step

Goal of this notebook: turn the messy raw columns into analysis-ready ones, **one decision at a time**, with the reasoning written down. Every step follows the same loop:

1. Look at real examples of the mess
2. Decide a rule
3. Write the smallest function that applies the rule
4. Verify it worked (don't just assume)

We never modify `data/raw/`. Everything here reads raw and writes to `data/processed/`.

In [1]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', 40)
df = pd.read_csv('../data/raw/Artworks.csv', encoding='utf-8-sig', low_memory=False)
df.shape

(160705, 30)

## Step 1 — Parenthesized fields: `Gender`, `Nationality`, `BeginDate`, `EndDate`

Look at the raw values:
```
Gender:       (male) (male) (male)
Nationality:  (American)
BeginDate:    (1886)     ← real year
BeginDate:    (0)        ← MoMA's placeholder for "unknown", not literally year 0
```

Two things going on:
- These are **multi-artist rows**: when an artwork has several artists, MoMA concatenates one `(value)` per artist. `Gender`, `Nationality`, `BeginDate`, `EndDate` all use this same convention.
- `(0)` in `BeginDate`/`EndDate` is a **placeholder for missing**, not a real date. Treating it as year 0 would silently corrupt every date-based analysis (imagine averaging birth years with a bunch of 0s mixed in).

**Decision:** write one general-purpose parser for the `(x) (y) (z)` pattern, reusable across all four columns, and make `(0)` map to `NaN` specifically for the date columns.

In [2]:
def split_parenthesized(value):
    """'(male) (male)' -> ['male', 'male'];  NaN -> []"""
    if pd.isna(value):
        return []
    return re.findall(r'\(([^)]*)\)', value)

# sanity check on a few raw examples before applying to the whole column
for v in ['(male)', '(male) (male) (male)', '(American)', '(0)', np.nan]:
    print(repr(v), '->', split_parenthesized(v))

'(male)' -> ['male']
'(male) (male) (male)' -> ['male', 'male', 'male']
'(American)' -> ['American']
'(0)' -> ['0']
nan -> []


In [3]:
# Apply to each column, producing a list-per-row
df['Gender_list'] = df['Gender'].apply(split_parenthesized)
df['Nationality_list'] = df['Nationality'].apply(split_parenthesized)
df['BeginDate_list'] = df['BeginDate'].apply(split_parenthesized)
df['EndDate_list'] = df['EndDate'].apply(split_parenthesized)

# For single-artist artworks (the vast majority), also keep a simple scalar version —
# most analyses don't need the full list, only the multi-artist ones do.
def first_or_nan(lst, treat_zero_as_missing=False):
    if not lst or lst[0] == '' or lst[0] is None:
        return np.nan
    val = lst[0]
    if treat_zero_as_missing and val == '0':
        return np.nan
    return val

df['Gender_primary'] = df['Gender_list'].apply(first_or_nan)
df['Nationality_primary'] = df['Nationality_list'].apply(first_or_nan)
df['BeginYear'] = df['BeginDate_list'].apply(lambda l: first_or_nan(l, treat_zero_as_missing=True))
df['EndYear'] = df['EndDate_list'].apply(lambda l: first_or_nan(l, treat_zero_as_missing=True))

df['BeginYear'] = pd.to_numeric(df['BeginYear'], errors='coerce')
df['EndYear'] = pd.to_numeric(df['EndYear'], errors='coerce')

df[['Gender_primary', 'Nationality_primary', 'BeginYear', 'EndYear']].head(8)

,Gender_primary,Nationality_primary,BeginYear,EndYear
0,male,Austrian,1841.0,1918.0
1,male,French,1944.0,NaN
2,male,Austrian,1876.0,1957.0
3,male,NaN,1944.0,NaN
4,male,Austrian,1876.0,1957.0
5,male,NaN,1944.0,NaN
6,male,NaN,1944.0,NaN
7,male,NaN,1944.0,NaN


In [4]:
# Verify: did we actually fix the mess, or just move it?
print('Gender_primary unique values:', df['Gender_primary'].unique())
print()
print('Any BeginYear == 0 left?', (df['BeginYear'] == 0).sum())
print('BeginYear range:', df['BeginYear'].min(), '-', df['BeginYear'].max())

Gender_primary unique values: <StringArray>
[                 'male',                'female',                     nan,
     'male (trans? ftm?', 'gender non-conforming',            'non-binary',
     'transgender woman',    'female (transwoman',     'woman, non-binary']
Length: 9, dtype: str

Any BeginYear == 0 left? 0
BeginYear range: 1730.0 - 2020.0


### The verification cell just caught a real bug

Look again at that `unique()` output above — mixed in with clean values like `'male'` and `'non-binary'` are mangled fragments: `'male (trans? ftm?'`, `'female (transwoman'`. Something broke. This is the entire point of verifying after applying a rule to the full column instead of just the handful of examples we tested it on — those examples happened to be the easy cases.

In [5]:
mask = df['Gender'].astype(str).str.contains('trans|non-binary|non-conforming', case=False, na=False, regex=True)
df.loc[mask, 'Gender'].unique()

<StringArray>
[                             '(male (trans? ftm?))',
                           '(gender non-conforming)',
                                      '(non-binary)',
 '(male) (female) (female) () (non-binary) (female)',
                               '(transgender woman)',
                             '(female (transwoman))',
                               '(woman, non-binary)']
Length: 7, dtype: str

**Root cause:** MoMA's `Gender` field uses parentheses two different ways at once — the outer `( )` delimits one artist's entry, but a handful of entries also have a *literal* parenthesis inside the description itself, e.g. `(male (trans? ftm?))` or `(female (transwoman))`. Our regex `\(([^)]*)\)` matches "everything up to the *first* `)`", so on `(male (trans? ftm?))` it grabbed `male (trans? ftm?` and silently truncated the identity description — while on the multi-artist case `(male) (female)` it worked fine because there's no nesting there.

This matters beyond just being a bug: it's exactly the kind of edge case that quietly mangles the data for an already-underrepresented group if nobody checks the tail of a `value_counts()`/`unique()` — a generic "looks fine" glance at the first few rows would never have caught it.

**Fix:** instead of matching each `(...)` independently, strip the one outer pair of parens and split on the boundary *between* artist entries — the literal `") ("` pattern — which only appears between separate artists, never inside one artist's description.

In [6]:
def split_parenthesized(value):
    """'(male) (male)' -> ['male', 'male'];  '(male (trans? ftm?))' -> ['male (trans? ftm?)']"""
    if pd.isna(value):
        return []
    s = value.strip()
    if s.startswith('(') and s.endswith(')'):
        s = s[1:-1]
    return re.split(r'\)\s*\(', s)

# re-check against both the easy cases and the ones that broke it before
for v in ['(male)', '(male) (male) (male)', '(male (trans? ftm?))',
          '(female (transwoman))', '(male) (female) (female) () (non-binary) (female)']:
    print(repr(v), '->', split_parenthesized(v))

'(male)' -> ['male']
'(male) (male) (male)' -> ['male', 'male', 'male']
'(male (trans? ftm?))' -> ['male (trans? ftm?)']
'(female (transwoman))' -> ['female (transwoman)']
'(male) (female) (female) () (non-binary) (female)' -> ['male', 'female', 'female', '', 'non-binary', 'female']


In [7]:
# Re-run Step 1 with the fixed parser
df['Gender_list'] = df['Gender'].apply(split_parenthesized)
df['Nationality_list'] = df['Nationality'].apply(split_parenthesized)
df['BeginDate_list'] = df['BeginDate'].apply(split_parenthesized)
df['EndDate_list'] = df['EndDate'].apply(split_parenthesized)

df['Gender_primary'] = df['Gender_list'].apply(first_or_nan)
df['Nationality_primary'] = df['Nationality_list'].apply(first_or_nan)
df['BeginYear'] = pd.to_numeric(df['BeginDate_list'].apply(lambda l: first_or_nan(l, treat_zero_as_missing=True)), errors='coerce')
df['EndYear'] = pd.to_numeric(df['EndDate_list'].apply(lambda l: first_or_nan(l, treat_zero_as_missing=True)), errors='coerce')

# verify: the truncated fragments should be gone, replaced by the full descriptions
sorted(df['Gender_primary'].dropna().unique())

['female',
 'female (transwoman)',
 'gender non-conforming',
 'male',
 'male (trans? ftm?)',
 'non-binary',
 'transgender woman',
 'woman, non-binary']

Notice `Gender_primary` still has an empty string `''` alongside `NaN` — that's `()` (an artist with no gender recorded) vs a genuinely missing cell. Worth normalizing both to `NaN` so `.isna()` catches everything consistently, since two different "missing" spellings is its own bug waiting to happen.

In [8]:
df['Gender_primary'] = df['Gender_primary'].replace('', np.nan)
df['Nationality_primary'] = df['Nationality_primary'].replace('', np.nan)
df['Gender_primary'].value_counts(dropna=False)

Gender_primary
male                     129263
female                    21475
NaN                        9847
female (transwoman)          62
non-binary                   42
male (trans? ftm?)           11
gender non-conforming         2
woman, non-binary             2
transgender woman             1
Name: count, dtype: int64

## Step 2 — Parsing `Date` (the artwork's creation year)

This one's messier: single years, ranges with three *different* dash characters (`-`, `–` en dash, `—` em dash), and free text like `"c. 1920"`. Regex on raw text before you decide a rule is how you find this — don't guess the format, look at samples first:

```
'1944—1952'   em dash range
'1931-1932'   hyphen range
'1927–28'     en dash, second year abbreviated
'1973'        plain year
```

**Decision:** for a range, use the **first** year as the artwork's `Year` — that's when creation started, which is the more standard convention in art-historical data. Document this choice explicitly since a different analysis might reasonably pick the midpoint or last year instead — it's a judgment call, not a fact.

In [9]:
def extract_year(value):
    """Pull the first 4-digit year found in free-text date strings."""
    if pd.isna(value):
        return np.nan
    match = re.search(r'\d{4}', str(value))
    return int(match.group()) if match else np.nan

for v in ['1973', '1944\u20141952', '1931-1932', '1927\u201328', 'c. 1920', 'n.d.']:
    print(repr(v), '->', extract_year(v))

'1973' -> 1973
'1944—1952' -> 1944
'1931-1932' -> 1931
'1927–28' -> 1927
'c. 1920' -> 1920
'n.d.' -> nan


In [10]:
df['Year'] = df['Date'].apply(extract_year)

# Validate: any implausible years? (MoMA opened 1929, collection includes older works, but nothing before ~1400 or after this year makes sense)
print('Year range:', df['Year'].min(), '-', df['Year'].max())
print('Missing Year:', df['Year'].isna().sum(), 'out of', len(df))
suspicious = df[(df['Year'] < 1400) | (df['Year'] > 2026)]
suspicious[['Title', 'Date', 'Year']].head(10)

Year range: 1768.0 - 3000.0
Missing Year: 4423 out of 160705


,Title,Date,Year
81433,Chopsticks,c. 3000 B.C.,3000.0


### Another one, caught by the same habit

The suspicious-range check above just flagged `Year == 3000` for "Chopsticks". Looking at the source: `Date` is `'c. 3000 B.C.'` — a genuinely ancient object, not a data error. Our `extract_year` function grabs the first 4-digit number and ignores the `B.C.` qualifier entirely, so it silently turned "3000 years before year 1" into "the year 3000" — a ~6000-year error from a single missed word.

Only one row is affected here, but the fix matters more as a habit than for this specific row: **a plausibility check on the output range is what surfaces this kind of silent unit error**, so it's worth handling properly rather than shrugging off one row.

In [11]:
def extract_year(value):
    """Pull the first 4-digit year; negate it if the string is tagged B.C."""
    if pd.isna(value):
        return np.nan
    s = str(value)
    match = re.search(r'\d{4}', s)
    if not match:
        return np.nan
    year = int(match.group())
    if re.search(r'B\.?C\.?', s, re.IGNORECASE):
        year = -year
    return year

df['Year'] = df['Date'].apply(extract_year)

print('Year range:', df['Year'].min(), '-', df['Year'].max())
suspicious = df[(df['Year'] < -10000) | (df['Year'] > 2026)]
print('Remaining suspicious rows:', len(suspicious))

Year range: -3000.0 - 2026.0
Remaining suspicious rows: 0


## Step 3 — Missing values: decide column by column, not globally

From the first notebook's missingness scan, the physical-dimension columns (`Weight (kg)`, `Circumference (cm)`, etc.) are 88–100% empty. That is **not a data quality problem** — most works are flat (prints, drawings, photos) and simply don't have a weight or circumference. Dropping those rows, or imputing a fake value, would be wrong — the absence itself is meaningful ("this is a 2D work").

Contrast that with `Medium` (5.5% missing) or `Gender_primary` (now ~9% missing after our parsing) — these *should* generally be knowable, so their absence is closer to a real gap, and should stay `NaN` rather than being silently dropped or filled, so downstream analysis can explicitly decide to exclude or footnote them.

**Decision:**
- Leave sparse *physical* columns as `NaN` — they're structurally, not accidentally, missing. Analyses that need them (e.g. "average sculpture weight") will naturally subset to non-null rows.
- Leave `Medium`, `Gender_primary`, `Nationality_primary`, `Year` as `NaN` too — never invent a value. Just make sure every downstream `groupby`/plot explicitly handles or reports the missing share, instead of silently excluding it.

In [12]:
for col in ['Medium', 'Gender_primary', 'Nationality_primary', 'Year']:
    pct = df[col].isna().mean() * 100
    print(f'{col:22s} {pct:5.1f}% missing')

Medium                   5.5% missing
Gender_primary           6.1% missing
Nationality_primary      4.5% missing
Year                     2.8% missing


## Step 4 — Duplicates

Check at the right grain: `ObjectID` should be unique (it's MoMA's own primary key) — if it isn't, that's a real problem, not a false alarm.

In [13]:
print('Duplicate ObjectID rows:', df['ObjectID'].duplicated().sum())

# .duplicated() needs hashable values, and our *_list columns hold Python lists — exclude them
hashable_cols = [c for c in df.columns if not c.endswith('_list')]
print('Fully duplicate rows:', df[hashable_cols].duplicated().sum())

Duplicate ObjectID rows: 0


Fully duplicate rows: 0


## Step 5 — `Classification`: a disguised missing value

`Classification` only has 42 unique values, so unlike free-text columns we can look at the *entire* distribution at once instead of sampling — and one entry stands out: `(not assigned)`, 686 rows.

That's not a real classification, it's MoMA's own placeholder for "we don't know." The problem: it's a **string**, not an empty cell, so `df['Classification'].isna()` reports 0% missing on this column — which is wrong. Anyone who ran a missingness check and trusted it would never find these 686 rows.

**Decision:** replace `'(not assigned)'` with `NaN` so missingness checks and any `groupby('Classification')` treat it consistently with every other kind of "we don't know."

In [14]:
# before: this looks fine, but we know it's wrong
print('Missing Classification (before):', df['Classification'].isna().sum())

df['Classification'] = df['Classification'].replace('(not assigned)', np.nan)

# after: now it reflects the real gap
print('Missing Classification (after): ', df['Classification'].isna().sum())

Missing Classification (before): 1
Missing Classification (after):  687


## Step 6 — Save the cleaned dataset

Save to `data/processed/`, never overwrite `data/raw/`. Keep both the original messy columns (`Date`, `Gender`, ...) *and* the new derived ones (`Year`, `Gender_primary`, ...) side by side — that way anyone reviewing this later can audit exactly how a cleaned value was derived from its source, instead of trusting it blindly.

In [15]:
import os
os.makedirs('../data/processed', exist_ok=True)

# list-type columns don't round-trip cleanly through CSV; drop them from the saved file
# (keep them in-session if you need multi-artist detail for a specific analysis)
save_cols = [c for c in df.columns if not c.endswith('_list')]
df[save_cols].to_csv('../data/processed/artworks_clean.csv', index=False)
print('Saved', len(df), 'rows to data/processed/artworks_clean.csv')

Saved 160705 rows to data/processed/artworks_clean.csv


## Recap: the decisions we made and why

| Column | Problem | Decision | Why |
|---|---|---|---|
| `Gender`, `Nationality`, `BeginDate`, `EndDate` | `(x) (y)` per-artist concatenation | Parse into lists + a `_primary` (first artist) scalar | Keeps both the detailed and the common-case-simple version |
| `Gender` | Nested parens in a few entries (`(male (trans? ftm?))`) truncated by the naive regex | Split on `") ("` between artists instead of matching each `(...)` independently | Caught only because we checked `unique()` on the full column, not just a sample — the naive version silently mangled minority-identity data |
| `BeginDate`/`EndDate` | `(0)` placeholder | Mapped to `NaN`, not year 0 | A literal 0 would silently poison any average/min/max |
| `Gender_primary` | `''` vs `NaN` — two missing spellings | Normalized both to `NaN` | One consistent way to test "is this missing" |
| `Date` | Free text, 3 dash types, `c. 1920` etc. | Regex-extract first 4-digit year into `Year` | Documented as a judgment call (first year of a range, not midpoint) |
| `Date` | `"c. 3000 B.C."` parsed as year 3000 (ignored the B.C.) | Negate the year when a B.C./BC tag is present | Caught by a plausibility check on the output range, not by reading every row |
| Physical dimension columns | 88–100% missing | Left as `NaN`, not dropped/imputed | Missingness is structural (most works are flat), not an error |
| `Classification` | `'(not assigned)'` is a placeholder string, not a real `NaN` | Replaced with `NaN` | `.isna()` was silently reporting 0% missing on 686 real gaps |
| `ObjectID` | Could have dupes | Checked, none found | Confirms the primary key is trustworthy |

**The general lesson:** several real bugs shipped in the *first* version of this cleaning code, and none were visible from a handful of examples or a naive `.isna()` call — they only surfaced because we checked the full `unique()` output, the full value range, and the full `value_counts()` afterward. That's the actual skill: not writing a rule that works on the cases you thought of, but verifying against the cases you didn't.